# 低通滤波器原理详解

## 为什么 SERDES 需要理解低通滤波器？

在高速串行传输中，**每一个物理信道本质上都是一个低通滤波器**：

- PCB 走线 → 低通特性
- 连接器 → 低通特性
- 电缆 → 低通特性

理解低通滤波器的工作原理，是理解 SERDES 信号完整性的基础。

---

## 本文内容

1. **什么是滤波器** - 基本概念
2. **频域视角** - 滤波器在频域做什么
3. **时域视角** - 滤波器在时域做什么
4. **四种经典滤波器** - Butterworth / Bessel / Chebyshev / 理想
5. **卷积与滤波** - 滤波的数学本质
6. **实际案例分析** - 方波通过低通滤波器

## 一、什么是滤波器？

### 直观理解

滤波器就像一个**筛子**，它允许某些频率的成分通过，阻挡其他频率的成分。

```
低通滤波器 (Low-pass): 允许低频通过，阻挡高频
高通滤波器 (High-pass): 允许高频通过，阻挡低频
带通滤波器 (Band-pass): 只允许某个频段通过
```

### 低通滤波器的频率响应

```
幅度响应 |H(f)|
    1 |-------\
      |        \
      |         \
    0 |----------\------
      0          fc    f
             截止频率
```

**关键参数**：
- **截止频率 (fc)**: 信号衰减到 -3dB (约 0.707 倍) 时的频率
- **阶数 (n)**: 决定过渡带的陡峭程度

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei']
plt.rcParams['axes.unicode_minus'] = False

print('库导入成功')

## 二、频域视角：滤波器在做什么？

### 傅里叶变换：从时域到频域

任何信号都可以分解为多个正弦波的叠加：

$$x(t) \leftrightarrow X(f)$$

滤波的本质：**在频域乘以滤波器的频率响应**

$$Y(f) = X(f) \cdot H(f)$$

其中：
- $X(f)$ = 输入信号的频谱
- $H(f)$ = 滤波器的频率响应
- $Y(f)$ = 输出信号的频谱

In [ ]:
# 演示：方波的频谱 + 低通滤波

fs = 1000  # 采样率
T = 1.0    # 时长
N = int(fs * T)
t = np.linspace(0, T, N, endpoint=False)

# 生成方波信号（包含多个频率分量）
f0 = 10  # 基频 10Hz
x = signal.square(2 * np.pi * f0 * t)

# 计算频谱
X = np.fft.fft(x)
freqs = np.fft.fftfreq(N, 1/fs)
magnitude = np.abs(X) / N

# 只取正频率
pos_mask = freqs >= 0
freqs_pos = freqs[pos_mask]
magnitude_pos = magnitude[pos_mask]

# 绘图
fig, axes = plt.subplots(2, 1, figsize=(12, 8))

# 时域
axes[0].plot(t[:200], x[:200], 'b-', linewidth=1)
axes[0].set_title('时域：方波信号', fontsize=12)
axes[0].set_xlabel('时间 (s)')
axes[0].set_ylabel('幅度')
axes[0].grid(True, alpha=0.3)

# 频域
axes[1].stem(freqs_pos[:50], magnitude_pos[:50], 'b', basefmt='k-', linefmt='b-', markerfmt='bo')
axes[1].set_title('频域：方波的频谱（只显示前 50 个频率分量）', fontsize=12)
axes[1].set_xlabel('频率 (Hz)')
axes[1].set_ylabel('幅度')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('观察要点：')
print('1. 方波由基频 (10Hz) + 奇次谐波 (30Hz, 50Hz, 70Hz...) 组成')
print('2. 谐波次数越高，幅度越小 (1/n 衰减)')
print('3. 低通滤波器会切除高频谐波，导致方波变圆滑')

## 三、时域视角：卷积

### 滤波的时域定义：卷积

在时域中，滤波等于**输入信号与滤波器冲激响应的卷积**：

$$y(t) = x(t) * h(t) = \int_{-\infty}^{\infty} x(\tau) \cdot h(t - \tau) \, d\tau$$

离散形式：

$$y[n] = \sum_{k=-\infty}^{\infty} x[k] \cdot h[n - k]$$

### 冲激响应 $h(t)$

冲激响应是滤波器对**单位冲激信号** $\delta(t)$ 的响应。

关键关系：

$$H(f) = \mathcal{F}\{h(t)\}$$

频率响应是冲激响应的傅里叶变换！

In [ ]:
# 演示：卷积如何产生滤波效果

# 理想低通滤波器的冲激响应是 sinc 函数
fc = 50  # 截止频率 50Hz
t_conv = np.linspace(-0.5, 0.5, 1000)

# sinc 函数：h(t) = 2*fc * sinc(2*fc*t)
h = 2 * fc * np.sinc(2 * fc * t_conv)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(t_conv, h, 'b-', linewidth=2)
ax.axvline(x=0, color='r', linestyle='--', alpha=0.5, label='t=0')
ax.set_title('理想低通滤波器的冲激响应 (sinc 函数)', fontsize=12)
ax.set_xlabel('时间 (s)')
ax.set_ylabel('幅度')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('观察要点：')
print('1. 理想低通的冲激响应是 sinc 函数')
print('2. sinc 函数有正负振荡，这就是滤波后出现振铃的原因')
print('3. 卷积就是输入信号与这个 sinc 函数"滑动相乘再求和"')

## 四、四种经典低通滤波器

| 类型 | 通带特性 | 过渡带 | 时域表现 | 典型应用 |
|------|----------|--------|----------|----------|
| **Butterworth** | 最平坦 | 中等 | 有过冲 | 通用滤波 |
| **Bessel** | 较平坦 | 较缓 | **无过冲** | 高速信号 |
| **Chebyshev** | 有纹波 | **最陡** | 大过冲 | 音频/测量 |
| **理想** | 完全平坦 | **无限陡** | 严重振铃 | 理论分析 |

### 核心权衡

```
过渡带越陡 → 时域过冲越大
时域越平滑 → 过渡带越缓

这是由傅里叶变换的不确定性原理决定的！
```

In [ ]:
# 对比四种滤波器的频率响应

fc = 50  # 截止频率
n = 4    # 阶数
f = np.linspace(0, 200, 1000)

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# 1. Butterworth
b_butt, a_butt = signal.butter(n, fc/(fs/2), btype='low')
w, H_butt = signal.freqs(b_butt, a_butt, w=f*np.pi*2)
axes[0,0].plot(f, 20*np.log10(np.abs(H_butt)), 'b')
axes[0,0].set_title('Butterworth: 通带最平坦', fontsize=11)
axes[0,0].set_xlabel('频率 (Hz)')
axes[0,0].set_ylabel('幅度 (dB)')
axes[0,0].grid(True, alpha=0.3)
axes[0,0].axvline(x=fc, color='r', linestyle='--', alpha=0.5, label='fc')
axes[0,0].legend()

# 2. Bessel
b_bes, a_bes = signal.bessel(n, fc/(fs/2), btype='low', norm='phase')
w, H_bes = signal.freqs(b_bes, a_bes, w=f*np.pi*2)
axes[0,1].plot(f, 20*np.log10(np.abs(H_bes)), 'g')
axes[0,1].set_title('Bessel: 过渡带最缓，但相位线性', fontsize=11)
axes[0,1].set_xlabel('频率 (Hz)')
axes[0,1].set_ylabel('幅度 (dB)')
axes[0,1].grid(True, alpha=0.3)
axes[0,1].axvline(x=fc, color='r', linestyle='--', alpha=0.5)

# 3. Chebyshev
b_che, a_che = signal.cheby1(n, 1, fc/(fs/2), btype='low')
w, H_che = signal.freqs(b_che, a_che, w=f*np.pi*2)
axes[1,0].plot(f, 20*np.log10(np.abs(H_che)), 'm')
axes[1,0].set_title('Chebyshev: 过渡带最陡，但通带有纹波', fontsize=11)
axes[1,0].set_xlabel('频率 (Hz)')
axes[1,0].set_ylabel('幅度 (dB)')
axes[1,0].grid(True, alpha=0.3)
axes[1,0].axvline(x=fc, color='r', linestyle='--', alpha=0.5)

# 4. 理想低通（理论）
H_ideal = np.where(f <= fc, 1, 0)
axes[1,1].plot(f, 20*np.log10(H_ideal + 1e-10), 'k')
axes[1,1].set_title('理想低通：无限陡的过渡带（物理不可实现）', fontsize=11)
axes[1,1].set_xlabel('频率 (Hz)')
axes[1,1].set_ylabel('幅度 (dB)')
axes[1,1].grid(True, alpha=0.3)
axes[1,1].axvline(x=fc, color='r', linestyle='--', alpha=0.5)
axes[1,1].set_ylim(-100, 5)

plt.tight_layout()
plt.show()

In [ ]:
# 对比四种滤波器对方波的时域响应

# 生成方波
f0 = 5  # 方波频率
x_square = signal.square(2 * np.pi * f0 * t)

# 滤波
y_butt = signal.filtfilt(b_butt, a_butt, x_square)
y_bes = signal.filtfilt(b_bes, a_bes, x_square)
y_che = signal.filtfilt(b_che, a_che, x_square)

fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(t[:500], x_square[:500], 'k--', linewidth=1, label='原始方波', alpha=0.5)
ax.plot(t[:500], y_butt[:500], 'b-', linewidth=2, label='Butterworth')
ax.plot(t[:500], y_bes[:500], 'g-', linewidth=2, label='Bessel')
ax.plot(t[:500], y_che[:500], 'm-', linewidth=2, label='Chebyshev')

ax.set_title('四种滤波器对方波的响应（截止频率 50Hz，4 阶）', fontsize=12)
ax.set_xlabel('时间 (s)')
ax.set_ylabel('幅度')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('观察要点：')
print('1. Butterworth: 上升沿较陡，有明显过冲')
print('2. Bessel: 上升沿最缓，几乎无过冲（最适合高速信号）')
print('3. Chebyshev: 上升沿最陡，过冲最大，振铃最严重')

## 五、Gibbs 现象：为什么滤波后会有振铃？

### 什么是 Gibbs 现象？

当用有限个谐波来近似一个**不连续信号**（如方波）时，在不连续点附近会出现**过冲和振荡**。

### 关键结论

$$
\text{过冲幅度} \approx 9\% \text{（与谐波数量无关）}
$$

即使使用无穷多个谐波，过冲仍然存在！

### 物理意义

Gibbs 现象揭示了**频域截断**与**时域振荡**之间的根本关系：

- 频域切得越陡 → 时域振荡越强
- 要减少时域振荡 → 必须放宽频带要求

In [ ]:
# 演示 Gibbs 现象：谐波数量对过冲的影响

f0 = 10  # 方波基频
t_gibbs = np.linspace(0, 0.2, 1000)

N_values = [1, 3, 5, 10, 20, 50, 100]

fig, ax = plt.subplots(figsize=(12, 6))

for N in N_values:
    y = np.zeros_like(t_gibbs)
    for n in range(1, N+1):
        k = 2*n - 1  # 奇次谐波
        y += (4/np.pi) * np.sin(2*np.pi*k*f0*t_gibbs) / k
    ax.plot(t_gibbs, y, label=f'N={N}个谐波', alpha=0.7)

# 理想方波
ideal = np.square(2*np.pi*f0*t_gibbs)
ax.plot(t_gibbs, ideal, 'k--', linewidth=2, label='理想方波', alpha=0.5)

ax.set_title('Gibbs 现象：谐波数量增加，过冲依然存在', fontsize=12)
ax.set_xlabel('时间 (s)')
ax.set_ylabel('幅度')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim(-1.5, 1.5)

plt.tight_layout()
plt.show()

print('观察要点：')
print('1. 谐波越多，上升沿越陡')
print('2. 但过冲幅度始终约 9%，不会消失')
print('3. 振荡的密度随谐波数量增加而增加')

## 六、SERDES 中的低通滤波器

### 信道 = 低通滤波器

在 SERDES 系统中，物理信道（PCB 走线、连接器、电缆）本质上是一个低通滤波器：

```
发送端方波 → 信道 (低通) → 接收端圆滑波形
```

### 眼图闭合的原因

1. **高频损耗** → 上升沿变缓 → 眼宽减小
2. **码间干扰 (ISI)** → 前一位影响后一位 → 眼高减小
3. **反射** → 阻抗不匹配 → 眼图畸变

### FFE/CTLE 的作用

- **FFE (发送端)**: 预加重，补偿高频损耗
- **CTLE (接收端)**: 均衡，提升高频分量

本质都是**反向补偿信道的低通特性**！

In [ ]:
# SERDES 信道仿真：方波通过低通信道

# 1. 生成伪随机比特序列
np.random.seed(42)
bits = np.random.randint(0, 2, 50)
samples_per_bit = 50
tx_signal = np.repeat(bits, samples_per_bit)

# 2. 模拟信道（Butterworth 低通）
fc_channel = 15  # 信道截止频率（相对单位）
b_ch, a_ch = signal.butter(3, fc_channel/(samples_per_bit/2), btype='low')
rx_signal = signal.lfilter(b_ch, a_ch, tx_signal)

# 3. 绘图
fig, axes = plt.subplots(2, 1, figsize=(14, 6))

# 时域对比
view_len = 20 * samples_per_bit
axes[0].plot(tx_signal[:view_len], 'b-', linewidth=1, label='发送端', alpha=0.7)
axes[0].plot(rx_signal[:view_len], 'r-', linewidth=2, label='接收端（经过信道）')
axes[0].set_title('SERDES 信道仿真：方波通过低通信道', fontsize=12)
axes[0].set_xlabel('采样点')
axes[0].set_ylabel('幅度')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 眼图
def plot_eye(sig, spb, ax):
    for i in range(0, len(sig)-2*spb, spb):
        ax.plot(sig[i:i+2*spb], color='blue', alpha=0.1)
    ax.set_title('眼图', fontsize=12)
    ax.grid(True, alpha=0.3)

plot_eye(rx_signal, samples_per_bit, axes[1])

plt.tight_layout()
plt.show()

print('观察要点：')
print('1. 接收端波形变圆滑，上升沿变缓')
print('2. 眼图张开度变小，但仍有采样窗口')
print('3. 如果信道损耗更大，眼图会完全闭合（无法正确采样）')

## 七、总结

### 核心公式回顾

| 域 | 运算 | 公式 |
|----|------|------|
| **频域** | 相乘 | $Y(f) = X(f) \cdot H(f)$ |
| **时域** | 卷积 | $y(t) = x(t) * h(t)$ |

### 滤波器选择原则

| 需求 | 推荐类型 |
|------|----------|
| 通带平坦 | Butterworth |
| 时域无过冲 | **Bessel** (SERDES 首选) |
| 过渡带陡 | Chebyshev |

### SerDes 启示

1. **信道是低通的** → 高频分量必然衰减
2. **方波需要高频谐波** → 经过信道后必然变圆滑
3. **FFE/CTLE 是补偿** → 反向提升高频分量
4. **眼图是评估指标** → 张开度决定能否正确采样

---

## 延伸阅读

- [低通滤波器对方波的影响](./低通滤波器对方波的影响.ipynb)
- [FFE 原理详解](./FFE 原理详解.ipynb)
- [SERDES 系统完整流程](./serdes 系统详解.ipynb)

![Colab](https://colab.research.google.com/assets/colab-badge.svg) [在 Colab 中打开](https://colab.research.google.com/github/yudonghai789/presonal_Blog/blob/master/高速传输 (serdes)/低通滤波器原理详解.ipynb)